In [1]:
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential, load_model
from tensorflow.keras.layers import Embedding, LSTM, Dense, Dropout
import pandas as pd
from sklearn.model_selection import train_test_split
import joblib
from sklearn.metrics import classification_report

In [2]:
df = pd.read_csv("../../dataset/Cleaned_Suicide_Detection_DL.csv")
df.columns

Index(['text', 'class', 'clean_text', 'labels'], dtype='object')

In [3]:
X = df["clean_text"]
y = df["labels"]

In [4]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [5]:
tokenizer = Tokenizer(num_words=5000)
tokenizer.fit_on_texts(X_train)

X_train_seq = tokenizer.texts_to_sequences(X_train)
X_test_seq = tokenizer.texts_to_sequences(X_test)

In [6]:
X_train_pad = pad_sequences(X_train_seq, maxlen=100)
X_test_pad = pad_sequences(X_test_seq, maxlen=100)

In [10]:
joblib.dump(tokenizer, "../../models/tokenizer.pkl")

['../../models/tokenizer.pkl']

In [20]:
model = Sequential([
    Embedding(input_dim=5000, output_dim=128),
    LSTM(64),
    Dense(1, activation="sigmoid")
])

In [21]:
model.compile(
    loss="binary_crossentropy",
    optimizer="adam",
    metrics=["accuracy"]
)

In [22]:
model.fit(
    X_train_pad, y_train,
    epochs=5,
    batch_size=32,
    validation_split=0.2
)

Epoch 1/5
4641/4641 ━━━━━━━━━━━━━━━━━━━━ 208s 44ms/step - accuracy: 0.9153 - loss: 0.2257 - val_accuracy: 0.9252 - val_loss: 0.2202
Epoch 2/5
4641/4641 ━━━━━━━━━━━━━━━━━━━━ 190s 41ms/step - accuracy: 0.9342 - loss: 0.1764 - val_accuracy: 0.9341 - val_loss: 0.1804
Epoch 3/5
4641/4641 ━━━━━━━━━━━━━━━━━━━━ 186s 40ms/step - accuracy: 0.9416 - loss: 0.1565 - val_accuracy: 0.9324 - val_loss: 0.1805
Epoch 4/5
4641/4641 ━━━━━━━━━━━━━━━━━━━━ 190s 41ms/step - accuracy: 0.9469 - loss: 0.1407 - val_accuracy: 0.9304 - val_loss: 0.1876
Epoch 5/5
4641/4641 ━━━━━━━━━━━━━━━━━━━━ 180s 39ms/step - accuracy: 0.9530 - loss: 0.1245 - val_accuracy: 0.9286 - val_loss: 0.1976


In [23]:
loss, acc = model.evaluate(X_test_pad, y_test)
print(acc)

1451/1451 ━━━━━━━━━━━━━━━━━━━━ 18s 12ms/step - accuracy: 0.9267 - loss: 0.2085
0.9266626238822937


In [24]:
model.save("../../models/lstm_model.h5")

In [7]:
model = load_model("../../models/lstm_model.h5")

sample = ["I dont want to die"]
sample = tokenizer.texts_to_sequences(sample)
sample = pad_sequences(sample, maxlen=100)

pred = model.predict(sample)[0][0]
pred

1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 719ms/step


np.float32(0.84429693)

In [10]:
y_pred = model.predict(X_test_pad)
y_pred = (y_pred > 0.5).astype(int)

1451/1451 ━━━━━━━━━━━━━━━━━━━━ 29s 20ms/step


In [11]:
print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

           0       0.92      0.94      0.93     23257
           1       0.94      0.91      0.93     23145

    accuracy                           0.93     46402
   macro avg       0.93      0.93      0.93     46402
weighted avg       0.93      0.93      0.93     46402

